# Spotify Tracks - Balanced Clustering & Quantile Normalization

This notebook demonstrates the implementation of **Quantile Normalization** for value unification and **Hungarian Balanced Clustering** to enforce equal-sized partitions.

These techniques prevent outlier soundtracks (e.g., extremely quiet or highly instrumental ambient tracks) from distorting distance-based clustering and ensure that clusters are balanced in size.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Data and Apply Stratified Downsampling

We load the Spotify Tracks dataset, clean it of missing/duplicate records, and extract a stratified sample of 3000 songs to keep the Hungarian assignment ($O(N^3)$ complexity) computationally efficient.

In [ ]:
data_path = "spotify_dataset.csv"
if not os.path.exists(data_path):
    data_path = "../spotify_dataset.csv"

if not os.path.exists(data_path):
    print("Error: spotify_dataset.csv not found.")
else:
    df = pd.read_csv(data_path)
    df = df.dropna(subset=['track_id', 'track_genre'])
    df = df.drop_duplicates(subset=['track_id'])
    
    # Stratified sampling
    sample_size = 3000
    genres = df['track_genre'].unique()
    samples_per_genre = max(1, int(sample_size / len(genres)))
    
    sampled_indices = []
    for g, group in df.groupby('track_genre'):
        sampled_indices.extend(group.sample(min(len(group), samples_per_genre), random_state=42).index)
    df_sampled = df.loc[sampled_indices].copy()
    df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)
    print("Stratified sample shape:", df_sampled.shape)

## 2. Preprocessing & Normalization Comparison

We compare standardizing our features using a `StandardScaler` versus normalizing them with a `QuantileTransformer` (Normal distribution). 

The Quantile Transformer forces features (like `loudness` and `instrumentalness`) to follow a Gaussian distribution, ensuring they share the same numeric ranges and scale shape, removing all extreme outliers.

In [ ]:
df_sampled['explicit'] = df_sampled['explicit'].astype(float)
features = [
    'popularity', 'duration_ms', 'danceability', 'energy', 'key', 
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 
    'liveness', 'valence', 'tempo', 'time_signature', 'explicit'
]
X = df_sampled[features].values

# Apply StandardScaler
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# Apply QuantileTransformer
qt = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)
X_q = qt.fit_transform(X)

# Plot comparisons for 3 select acoustic features
compare_features = ['loudness', 'instrumentalness', 'tempo']
fig, axes = plt.subplots(3, 2, figsize=(12, 12))

for i, feat in enumerate(compare_features):
    idx = features.index(feat)
    
    # Standard Scaled
    sns.histplot(X_std[:, idx], ax=axes[i, 0], kde=True, color='skyblue')
    axes[i, 0].set_title(f"{feat} (StandardScaler)")
    
    # Quantile Transformed
    sns.histplot(X_q[:, idx], ax=axes[i, 1], kde=True, color='salmon')
    axes[i, 1].set_title(f"{feat} (QuantileTransformer - Normal)")

plt.tight_layout()
plt.show()

## 3. Implement Hungarian Balanced KMeans

We implement the `HungarianBalancedKMeans` class inline. This model runs standard KMeans to initialize the centroids, then iteratively uses the Hungarian algorithm (`linear_sum_assignment`) to match points to centroids under equal-size constraints.

In [ ]:
class HungarianBalancedKMeans:
    def __init__(self, n_clusters=5, max_iter=20, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.cluster_centers_ = None
        self.labels_ = None

    def fit(self, X):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        K = self.n_clusters

        # Initialize centroids
        kmeans = KMeans(n_clusters=K, random_state=self.random_state, n_init=5)
        kmeans.fit(X)
        centroids = kmeans.cluster_centers_.copy()

        # Enforced cluster sizes
        sizes = [n_samples // K + (1 if i < n_samples % K else 0) for i in range(K)]
        cluster_mapping = []
        for i in range(K):
            cluster_mapping.extend([i] * sizes[i])
        cluster_mapping = np.array(cluster_mapping)

        labels = -np.ones(n_samples, dtype=int)

        for iteration in range(self.max_iter):
            # Replicate centroids
            repeated_centroids = []
            for i in range(K):
                repeated_centroids.extend([centroids[i]] * sizes[i])
            repeated_centroids = np.array(repeated_centroids)

            # Distance Matrix
            C = cdist(X, repeated_centroids, metric='euclidean')

            # Bipartite matching via Hungarian algorithm
            row_ind, col_ind = linear_sum_assignment(C)

            new_labels = np.zeros(n_samples, dtype=int)
            new_labels[row_ind] = cluster_mapping[col_ind]

            changes = np.sum(new_labels != labels)
            print(f"Iteration {iteration+1}: {changes} label assignments changed")
            
            if np.array_equal(new_labels, labels):
                break
            labels = new_labels

            # Recalculate centroids
            new_centroids = np.zeros_like(centroids)
            for i in range(K):
                cluster_points = X[labels == i]
                if len(cluster_points) > 0:
                    new_centroids[i] = cluster_points.mean(axis=0)
                else:
                    new_centroids[i] = X[np.random.choice(n_samples)]
            centroids = new_centroids

        self.cluster_centers_ = centroids
        self.labels_ = labels
        return self

## 4. Run Clustering and Compare Sizes & Metrics

We run standard KMeans and Hungarian Balanced KMeans on the quantile-transformed features and compare the size distribution and geometric validation metrics.

In [ ]:
n_clusters = 5

# Standard KMeans
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_q)

# Hungarian Balanced KMeans
hb_kmeans = HungarianBalancedKMeans(n_clusters=n_clusters, max_iter=20, random_state=42)
hb_kmeans.fit(X_q)
hb_labels = hb_kmeans.labels_

# Sizes
km_sizes = np.bincount(kmeans_labels, minlength=n_clusters)
hb_sizes = np.bincount(hb_labels, minlength=n_clusters)

print("\n--- Cluster Sizes Comparison ---")
for i in range(n_clusters):
    print(f"Cluster {i}: Standard KMeans size = {km_sizes[i]} | Hungarian KMeans size = {hb_sizes[i]}")

# Metrics
print("\n--- Metrics Comparison ---")
print("Standard KMeans:")
print(f"  Silhouette: {silhouette_score(X_q, kmeans_labels):.4f}")
print(f"  Davies-Bouldin: {davies_bouldin_score(X_q, kmeans_labels):.4f}")
print("Hungarian Balanced KMeans:")
print(f"  Silhouette: {silhouette_score(X_q, hb_labels):.4f}")
print(f"  Davies-Bouldin: {davies_bouldin_score(X_q, hb_labels):.4f}")

## 5. Visualize Projections in PCA Space

We visualize both partitions in a 2D PCA subspace.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_q)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors = plt.cm.get_cmap('tab10', n_clusters)

# Plot Standard KMeans
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=[colors(l) for l in kmeans_labels], s=15, alpha=0.8)
axes[0].set_title("Standard KMeans (Unbalanced)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].grid(True, alpha=0.3)

# Plot Hungarian Balanced KMeans
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=[colors(l) for l in hb_labels], s=15, alpha=0.8)
axes[1].set_title("Hungarian Balanced KMeans (Perfect Equal Sizes)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Profile the Balanced Clusters

Let's look at the original features' mean values to understand the acoustic nature of the balanced clusters.

In [ ]:
df_sampled['hungarian_balanced_cluster'] = hb_labels
profiles = df_sampled.groupby('hungarian_balanced_cluster')[features].mean()
profiles